In [1]:
# display warnings
import pandas as pd
from IPython.display import display, HTML
import warnings

# enable scrolling in large outputs
def show_scroll(df, height=400):
    display(HTML(f'<div style="height:{height}px; overflow:auto">{df.to_html()}</div>'))

pd.set_option("display.max_rows", 200)       # kaç satır göstersin
pd.set_option("display.max_colwidth", None)  # left/right kırpılmasın
warnings.filterwarnings("ignore", category=UserWarning)

In [4]:
import re 
from collections import Counter

CAPS_BLOCK = r"\b(?:[A-ZÇĞİÖŞÜ]{2,}\s+)+[A-ZÇĞİÖŞÜ]{2,}\b"
df = pd.read_csv(r"C:\Users\user\Desktop\chp_report\chp_report\data\corpus_main_processed.csv")

##########################
def remove_caps_blocks(text):
    return re.sub(CAPS_BLOCK, "", text)

df["text_nocaps"] = df["text_bert"].apply(remove_caps_blocks)

#########################
# inspect this later to complete the list
dont_touch = ["Dr.", "Prof.", "Doç.", "Hz.", "Sn.", "T.C.", "A.Ş.", "M.I.T",
              "Av.", "vs.", "vb.", "M.", "İ."]

def split_sentences(text):
    """"""
    for abbr in dont_touch:
        # \bdont_touch\. (e.g. \bDr\.)
        text = re.sub(r"\b" + re.escape(abbr), abbr.replace(".", "<DOT>"), text)
    pieces = re.split(r"[.?!…]+(?=\s|$)", text) # handles examples such as M.I.T.'ye, T.C.'nin 
    sentences = [piece.strip().replace("<DOT>", ".") for piece in pieces if piece.strip()]
    return sentences

df["sentences"] = df["text_nocaps"].apply(split_sentences)

#####################
def remove_short(sentences, min_words=4): # updated as 4, see the next cell
    return [s for s in sentences if len(s.split()) >= min_words]

df["sentences_long"] = df["sentences"].apply(remove_short)
    

#df[["text_bert", "n_sentences"]].head()
#show_scroll(df[["text_nocaps", "n_sentences"]])
df["n_sentences"] = df["sentences"].apply(len)
a = df["n_sentences"].sum()
df["n_sentences_long"] = df["sentences_long"].apply(len)
b = df["n_sentences_long"].sum()

print(f"before removing short sentences: {a}")
print(f"after removing short sentences: {b}")

# (df["n_sentences_long"] == 0).sum() = 1
sent_df = df.explode("sentences_long").dropna(subset=["sentences_long"])
len(sent_df)

sent_df = sent_df.reset_index().rename(columns={"index": "doc_id"})
sent_df = sent_df[["id", "date", "speaker", "venue", "sentences_long"]]
docs = sent_df["sentences_long"].tolist()
sent_df.columns


before removing short sentences: 602605
after removing short sentences: 515174


Index(['id', 'date', 'speaker', 'venue', 'sentences_long'], dtype='str')

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

def turkish_lower(text):
    return text.replace("I", "ı").replace("İ", "i").lower()

with open(r"C:\Users\user\Desktop\chp_report\chp_report\data\stopwords_tr.txt", encoding="utf-8") as f:
    stopwords_tr = [turkish_lower(w.strip()) for w in f if w.strip()]

vectorizer = CountVectorizer(stop_words=stopwords_tr, min_df=10, preprocessor=turkish_lower)

model = SentenceTransformer('emrecan/bert-base-turkish-cased-mean-nli-stsb-tr', device="cuda")

embeddings = model.encode(docs, batch_size=64, show_progress_bar=True)
np.save("embeddings_emrecan.npy", embeddings)

topic_model = BERTopic(embedding_model=model, vectorizer_model=vectorizer)
topics, probs = topic_model.fit_transform(docs, embeddings)




Batches: 100%|██████████| 8050/8050 [05:51<00:00, 22.91it/s]


# stopwordleri konuları okuyarak güncelle, embeddings py'daki kelimeleri de al

In [8]:
topic_model.get_topic_info()


,Topic,Count,Name,Representation,Representative_Docs
0,-1,248960,-1_erdoğan_hayır_asla_kadın,"[erdoğan, hayır, asla, kadın, sahip, meclis, insanlar, sorunları, milletin, siyasi]","[Ya ben milleti temsil ediyorum, vatandaş geliyor bana oy veren vatandaş geliyor, Benim derdimi çöz diyor, Ayrıca benzer sorunu diyelim ki bir başka belediye yaşıyorsa oradaki arkadaşlarınıza telefon edip biz bir proje hazırladık, oldukça verimli sonuçlar elde ettik, vatandaş son derece mutlu ama bize yansıdığı kadarıyla sizin beldenizde de böyle bir sorun var, Hani diyorlar ya, ‘Cumhuriyet Halk Partisi belediyeler ne yapıyor, ne yapacak?’ Geçen seçimlerden önce dediler ki ‘Denizli’de CHP’ye vermeyin, o gelirse sosyal yardımları keser, yoksullara verilenleri vermez, hizmetler aksar.’ Geldik ve başkan onların yaptığı gibi hiç maaşı olmayana değil, asgari ücretin üçte birinin altında kalana değil, bırak asgari ücreti, iki asgari ücretin altında geliri olan haneyi yoksul biliyor, yoksul sayıyor]"
1,0,1395,0_söyledim_söyledik_defalarca_söyleyeceğim,"[söyledim, söyledik, defalarca, söyleyeceğim, söylüyoruz, gittiğiniz, söyleyin, söylemeye, laf, söylerim]","[Onun için defalarca söyledim, Bütün bunları defalarca söyledik, defalarca ifade ettik, Bunu söyledim, defalarca da söyledim]"
2,1,1297,1_özgür_katıldı_özel_lideri,"[özgür, katıldı, özel, lideri, gerçekleştirilen, konuşan, düzenlenen, konuşmada, görüşmenin, ilçesinde]","[- Özgür Özel'in soyunu sopunu arıyorlarsa, Cumhuriyet Halk Partisi Lideri Özgür Özel, Manisa’nın Salihli ilçesindeki halk buluşmasına katıldı, Cumhuriyet Halk Partisi Lideri Özgür Özel, Halk TV yayınına katıldı]"
3,2,1250,2_sandığa_elinizi_vicdanınıza_kullanın,"[sandığa, elinizi, vicdanınıza, kullanın, oyumuzu, oyunuzu, kullanacağız, sandığı, giderken, düşünerek]","[Sadece sandığa gidin, elinizi vicdanınıza koyun ve oyunuzu öyle kullanın, Sandığa giderken elinizi vicdanınıza koyup oyunuzu öyle kullanın, Sandığa giderken elinizi vicdanınıza koyun ve oyunuzu öyle kullanın]"
4,3,1143,3_adaleti_kavramdır_adalet_adaletten,"[adaleti, kavramdır, adalet, adaletten, hukuku, adaletin, adalete, kainat, soylu, sağlayacağız]","[Bu ülkeye hakkı, hukuku ve adaleti hep birlikte getireceğiz, O da adaleti savunacaktır, O hep adaleti savunmuştur]"
...,...,...,...,...,...
5460,5459,10,5459_bilgiye_dayalı_doğruydu_söylediğimiz,"[bilgiye, dayalı, doğruydu, söylediğimiz, muhalefet, devletimizin, payda, kavgasız, zemini, dikkati]","[Evet bunlar bilgiye dayalı muhalefet yapıyorlar, Bilgiye dayalı muhalefet bilgiye dayalı, Bilgiye dayalı muhalefet bizim demokrasi çıtamızı yükseltir]"
5461,5460,10,5460_hasan_ertesinde_çıkacağına_tatlı,"[hasan, ertesinde, çıkacağına, tatlı, dolaşıyor, inandık, vardık, kaldırıyoruz, adıma, ankete]","[Benimle birlikte gelen yol arkadaşım sevgili Veli Ağbaba dedi ki ‘Keşke bu Hasan gibi genç biri il başkanı olsa da partide bir gençleşme, dinamizm olsa.’ Dedim ki ‘Veli abi Hasan olacak.’ Dedi ki ‘Nereden olacak?’ ‘Ben’ dedim, ‘Salona baktım; eşi Güneş’le birlikte el ele dolaşıyor, Ben inanıyorum Hasan buradan il başkanı olarak çıkacak.’ Ankara’ya vardık, varmadık haber geldi ve Hasan, İl Başkanı oldu, Sonra seçimler geldi, Hasan milletvekili adayı oldu]"
5462,5461,10,5461_kavuşacak_başaran_başkenti_yazıldığı,"[kavuşacak, başaran, başkenti, yazıldığı, bugünde, mızın, attığınız, görüşlerden, basan, başlatan]","[Ankara Türkiye Cumhuriyeti devletinin başkenti, Ankara Milli Kurtuluş Savaşının merkezi, Ankara, TBMM’yi yani Gazi Meclisi bağrına basan Ankara, Ankara kimliğine kavuşacak, Kuvayı Milliye kimliğine kavuşacak, Ankara’yı yıllardır o bayrağı onurla ve gururla dalgalandıran Ankara’daki ilk değişimi başlatan, tüm siyasi görüşlerden oy almayı başaran kahramanı var burada]"
5463,5462,10,5462_kyk_verebilirdik_kredisi_öğrencilere,"[kyk, verebilirdik, kredisi, öğrencilere, gelirli, işsizlere, burs, öğrencisine, dar, harcanacak]","[İşsizlere ayda 15 bin lira işsizlik maaşı verebileceğin, tüm öğrencilere 3 bin lira değil, 30 bin lira KYK kre

In [75]:
#df.isna().sum() # 0 empty row

# any caps_block left?
#df["text_nocaps"].str.contains(r"[^.?!…\s]\s*" + CAPS_BLOCK).sum()

# test one example for the regex pattern
#test = " belki o zaman bilmiyorum… Hani konuşulanlar, Lazkiye'de onlara da bir otonom bölge sağlanıp, Esad’ın da güvenliğinin sağlandığı bir bölge teklif edilerek iktidarı devretmesi, kendisinin -belki bugün olduğu gibi ama- daha onurlu bir çıkışla ülkeyi terk edip Suriye’de bir geçiş hükümetine, demokrasiye doğru adım atmasını göze alsaydı bu hale gelmezdi, ülkesinden böyle kaçmazdı. Biz Suriye’ye sadece demokrasi telkin ettik. Ben Suriye’de Baas rejimi sürsün filan ister miyim? Ben niye isteyeyim yani? Ne hissedeceğim."
#re.split(r"[.?!…]+", test)

# inspection
#df["n_sentences"].describe()

# check dropped sentences (shorter than 5 words)
## update = 4 words still matter, lower the threshold to 4
dropped = df["sentences"].explode()
dropped = dropped[dropped.str.split().str.len() < 4] # 4
dropped.sample(30, random_state=4)

1404                                  Yok
785                                   Suç
169                   Tercihi kim yapacak
477                          İntihar etti
32                                   Tabi
1000               Seçim heyecanı yaşıyor
209                Uğramaması ayıp olurdu
1546                      Yıl Cami burası
728                    Ne kadar istiyoruz
390                        Ruhu şad olsun
482                    Ne oldu bilmiyorum
875               Söz veriyorum çözeceğim
749                       Gençler oynuyor
946                  Hep beraber olacağız
1517    Normali, siyasilerin konuşmasıdır
445                        Neden bu düzen
997                      İttifaklar vardı
929                                     4
179                                  Öyle
912                                  Asla
497     Milletimizle beraber kurtaracağız
635                       Bunları anlatın
1121                           Yapıyorlar
1255                 İstanbul’a so